## 1. Import Libraries và Cấu hình

In [ ]:
import pandas as pd
import numpy as np
import time
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ML Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# OpenAI SDK (dùng cho API bên thứ 3)
from openai import OpenAI

# --- CẤU HÌNH ---
# ⚠️ THAY API KEY CỦA BẠN VÀO ĐÂY (API key từ pinkyne.com)
API_KEY = "sk-Aibx5xty0Bd1qwmuN4IQ6JQMUg6suCAXOxYdMA2EOjyBEwh9"  # <-- Thay bằng API key từ pinkyne.com
BASE_URL = "https://vpi.pinkyne.com/v1"

# Model Gemini (qua API bên thứ 3)
MODEL_NAME = "gemini-2.5-flash"  # Hoặc tên model tương ứng trên pinkyne

# File paths
INPUT_CSV = "eval_level2_1000.csv"
OUTPUT_CSV = "eval_gemini_level2_results.csv"
CHECKPOINT_CSV = "eval_gemini_level2_checkpoint.csv"  # File lưu tiến trình

# === RATE LIMIT CONFIG (API bên thứ 3 thường ít limit hơn) ===
BATCH_SIZE = 20                    # Tăng lên vì ít bị limit hơn
DELAY_BETWEEN_REQUESTS = 2.0       # Giảm xuống vì API bên thứ 3
DELAY_BETWEEN_BATCHES = 30.0       # Giảm xuống
MAX_RETRIES = 5                    # Số lần retry khi gặp lỗi
RETRY_DELAY = 30                   # Giây chờ khi retry

# Sampling (đặt None để chạy toàn bộ)
SAMPLE_SIZE = None  # None = chạy toàn bộ dataset
START_INDEX = 0     # Bắt đầu từ index nào (tự động skip nếu có checkpoint)

print("✅ Import thành công!")
print(f"📌 Model: {MODEL_NAME}")
print(f"📌 Base URL: {BASE_URL}")
print(f"📌 Dataset: {INPUT_CSV}")
print(f"\n⚙️ Rate Limit Config (API bên thứ 3):")
print(f"   - Batch size: {BATCH_SIZE} requests/batch")
print(f"   - Delay giữa requests: {DELAY_BETWEEN_REQUESTS}s")
print(f"   - Delay giữa batches: {DELAY_BETWEEN_BATCHES}s")
print(f"   - Max retries: {MAX_RETRIES}, retry delay: {RETRY_DELAY}s")

## 2. Khởi tạo OpenAI Client (API bên thứ 3)

In [ ]:
# Hàm tạo/reset client
def create_client():
    """Tạo mới OpenAI client với base URL của bên thứ 3"""
    return OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL
    )

# Khởi tạo client lần đầu
client = create_client()

# Test kết nối
try:
    test_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": "Xin chào, bạn là ai?"}
        ],
        max_tokens=100
    )
    print("✅ Kết nối API thành công!")
    
    # Xử lý trường hợp response trả về string hoặc object
    if hasattr(test_response, 'choices'):
        content = test_response.choices[0].message.content
    else:
        content = str(test_response)
        
    print(f"📝 Test response: {content[:100]}...")
except Exception as e:
    print(f"❌ Lỗi kết nối: {e}")
    print("⚠️ Vui lòng kiểm tra lại API key và Base URL!")

## 3. Định nghĩa hàm gọi API (OpenAI SDK)

In [ ]:
def call_api_with_retry(messages, max_retries=MAX_RETRIES):
    """
    Gọi API với xử lý retry tự động khi gặp lỗi.
    Sử dụng OpenAI SDK với base URL của bên thứ 3.
    """
    global client
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                max_tokens=500,
                temperature=0.3  # Giảm temperature để kết quả ổn định hơn
            )
            return response.choices[0].message.content
            
        except Exception as e:
            error_msg = str(e).lower()
            
            # Xử lý rate limit
            if "rate" in error_msg or "quota" in error_msg or "429" in error_msg:
                wait_time = RETRY_DELAY * (attempt + 1)
                print(f"\n⚠️ Rate limit! Đợi {wait_time}s rồi retry ({attempt+1}/{max_retries})...")
                time.sleep(wait_time)
                
                # Reset client
                print("🔄 Reset client...")
                client = create_client()
                time.sleep(2)
                continue
            
            # Resource exhausted
            if "resource" in error_msg or "exhausted" in error_msg:
                wait_time = 120  # 2 phút
                print(f"\n🔴 Resource exhausted! Đợi {wait_time}s...")
                time.sleep(wait_time)
                client = create_client()
                continue
            
            # Các lỗi khác
            if attempt < max_retries - 1:
                wait_time = RETRY_DELAY
                print(f"\n⚠️ Lỗi: {e}. Đợi {wait_time}s rồi retry ({attempt+1}/{max_retries})...")
                time.sleep(wait_time)
                continue
            else:
                return f"[ERROR] {str(e)}"
    
    return "[ERROR] Max retries exceeded"


def ask_true_false(question):
    """
    Hỏi một câu hỏi Đúng/Sai qua API.
    """
    messages = [
        {
            "role": "system",
            "content": """Bạn là chuyên gia về điện ảnh và người nổi tiếng. 
Nhiệm vụ: Xác định mệnh đề là ĐÚNG hay SAI.
QUY TẮC:
- Nếu mệnh đề đúng theo kiến thức của bạn → Trả lời A (Đúng)
- Nếu mệnh đề sai hoặc bạn không chắc chắn → Trả lời B (Sai)
Trả lời theo format: "Đáp án: [A/B]. Lý do: [giải thích ngắn gọn]"
"""
        },
        {
            "role": "user",
            "content": f"""CÂU HỎI: {question}

A) Đúng
B) Sai

Hãy chọn đáp án:"""
        }
    ]
    
    return call_api_with_retry(messages)


print("✅ Định nghĩa hàm gọi API thành công!")

## 4. Định nghĩa hàm đánh giá câu trả lời Đúng/Sai

In [ ]:
def evaluate_true_false_answer(bot_answer, ground_truth_letter):
    """
    Đánh giá câu trả lời Đúng/Sai của Gemini.
    
    Args:
        bot_answer: Câu trả lời của Gemini
        ground_truth_letter: "A" (Đúng) hoặc "B" (Sai)
    
    Returns: 1 nếu đúng, 0 nếu sai
    """
    # Xử lý trường hợp lỗi
    if bot_answer.startswith("[ERROR]"):
        return 0
    
    bot_ans_lower = bot_answer.lower()
    
    # Xác định bot chọn A hay B
    bot_choice = None
    
    # Pattern cho đáp án A (Đúng) - ưu tiên pattern rõ ràng
    patterns_a_strong = [
        "đáp án: a", "đáp án a", "chọn a", "[a]", ": a.", ": a)",
        "đáp án là a", "trả lời: a", "answer: a", "→ a", "=> a"
    ]
    
    # Pattern cho đáp án B (Sai) - ưu tiên pattern rõ ràng
    patterns_b_strong = [
        "đáp án: b", "đáp án b", "chọn b", "[b]", ": b.", ": b)",
        "đáp án là b", "trả lời: b", "answer: b", "→ b", "=> b"
    ]
    
    # Ưu tiên check pattern A/B rõ ràng trước
    for pattern in patterns_a_strong:
        if pattern in bot_ans_lower:
            bot_choice = "A"
            break
    
    if bot_choice is None:
        for pattern in patterns_b_strong:
            if pattern in bot_ans_lower:
                bot_choice = "B"
                break
    
    # Nếu chưa xác định, check keyword
    if bot_choice is None:
        # Keywords cho đáp án đúng
        keywords_true = ["đúng", "chính xác", "có", "phải", "xác nhận", "correct", "true", "yes"]
        # Keywords cho đáp án sai
        keywords_false = ["sai", "không đúng", "không phải", "không có", "không", "incorrect", "false", "no"]
        
        # Đếm keyword
        a_score = sum(1 for k in keywords_true if k in bot_ans_lower)
        b_score = sum(1 for k in keywords_false if k in bot_ans_lower)
        
        # Bonus cho các phrase mạnh
        if "không có liên" in bot_ans_lower or "không tìm thấy" in bot_ans_lower:
            b_score += 2
        if "có liên quan" in bot_ans_lower or "có mối quan hệ" in bot_ans_lower:
            a_score += 2
        
        if a_score > b_score:
            bot_choice = "A"
        elif b_score > a_score:
            bot_choice = "B"
        else:
            bot_choice = "B"  # Default: Sai (conservative)
    
    # So sánh với ground truth
    return 1 if bot_choice == ground_truth_letter else 0


print("✅ Định nghĩa hàm đánh giá Đúng/Sai thành công!")

## 5. Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv(INPUT_CSV)
print(f"📊 Đã tải {len(df)} câu hỏi từ {INPUT_CSV}")
print(f"\n📋 Phân bố theo loại quan hệ:")
print(df['rel_type'].value_counts())

print(f"\n📋 Phân bố đáp án:")
print(df['answer'].value_counts())

# Preview
df.head()

In [ ]:
# Lấy dữ liệu TUẦN TỰ (không random) để dễ resume
if SAMPLE_SIZE:
    df_eval = df.iloc[START_INDEX:START_INDEX + SAMPLE_SIZE].reset_index(drop=True)
    print(f"⚠️ Chạy trên {len(df_eval)} câu hỏi (từ index {START_INDEX} đến {START_INDEX + SAMPLE_SIZE - 1})")
else:
    df_eval = df.iloc[START_INDEX:].reset_index(drop=True)
    print(f"🚀 Chạy đánh giá trên {len(df_eval)} câu hỏi (từ index {START_INDEX})")

# Tính số batch
num_batches = (len(df_eval) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"\n📊 Chia thành {num_batches} batches, mỗi batch {BATCH_SIZE} câu")

# Ước tính thời gian
time_per_batch = BATCH_SIZE * DELAY_BETWEEN_REQUESTS + DELAY_BETWEEN_BATCHES
estimated_time = num_batches * time_per_batch / 60
print(f"⏱️ Thời gian ước tính: ~{estimated_time:.1f} phút (~{estimated_time/60:.1f} giờ)")

# AUTO RESUME: Tự động tiếp tục từ checkpoint nếu có
if os.path.exists(CHECKPOINT_CSV):
    checkpoint_df = pd.read_csv(CHECKPOINT_CSV)
    print(f"\n💾 Tìm thấy checkpoint với {len(checkpoint_df)} kết quả đã lưu")
    print("🔄 Tự động tiếp tục từ checkpoint...")
    resume = True
else:
    checkpoint_df = None
    resume = False
    print("\n🆕 Bắt đầu mới từ đầu...")

## 6. Chạy Đánh giá Gemini - Level 2

In [ ]:
# === HOÀN TOÀN TỰ ĐỘNG - KHÔNG CẦN CAN THIỆP ===

# Khởi tạo hoặc load từ checkpoint
if resume and checkpoint_df is not None:
    results = checkpoint_df.to_dict('records')
    start_idx = len(results)
    print(f"📂 Tự động resume từ câu hỏi thứ {start_idx + 1}/{len(df_eval)}")
else:
    results = []
    start_idx = 0

# Đếm số lỗi
error_count = 0
current_batch = start_idx // BATCH_SIZE

# Thời gian bắt đầu
start_time = time.time()

print(f"\n{'='*60}")
print(f"🌙 BẮT ĐẦU ĐÁNH GIÁ - LEVEL 2 (True/False)")
print(f"{'='*60}")
print(f"🤖 Model: {MODEL_NAME} (via {BASE_URL})")
print(f"📋 Từ câu {start_idx + 1} đến câu {len(df_eval)}")
print(f"📋 Loại câu hỏi: Đúng/Sai (True/False)")
print(f"📋 Config: {BATCH_SIZE} req/batch, {DELAY_BETWEEN_REQUESTS}s delay")
print(f"📋 Checkpoint tự động lưu sau mỗi batch")
print(f"{'='*60}\n")

# === VÒNG LẶP ĐÁNH GIÁ CHÍNH ===
for idx in tqdm(range(start_idx, len(df_eval)), desc="Đánh giá Level 2", initial=start_idx, total=len(df_eval)):
    row = df_eval.iloc[idx]
    question = row['question']
    ground_truth_letter = row['answer']  # "A" hoặc "B"
    ground_truth_text = row['ground_truth']  # Giải thích
    rel_type = row['rel_type']
    
    # Label: A = 1 (Đúng), B = 0 (Sai)
    ground_truth_label = 1 if ground_truth_letter == "A" else 0
    
    # Kiểm tra nếu là đầu batch mới (trừ batch đầu tiên)
    batch_num = idx // BATCH_SIZE
    if batch_num > current_batch:
        current_batch = batch_num
        
        # Tính tiến độ
        elapsed = time.time() - start_time
        progress = (idx - start_idx) / (len(df_eval) - start_idx) * 100 if len(df_eval) > start_idx else 0
        remaining = (elapsed / (idx - start_idx + 1)) * (len(df_eval) - idx) if idx > start_idx else 0
        
        print(f"\n\n{'='*50}")
        print(f"🔄 BATCH {batch_num + 1}/{num_batches} | Tiến độ: {progress:.1f}%")
        print(f"⏱️ Đã chạy: {elapsed/60:.1f} phút | Còn lại: ~{remaining/60:.1f} phút")
        print(f"⏳ Nghỉ {DELAY_BETWEEN_BATCHES}s...")
        print(f"{'='*50}")
        
        time.sleep(DELAY_BETWEEN_BATCHES)
        
        # Reset client định kỳ
        client = create_client()
        time.sleep(2)
        print("✅ Đã reset client, tiếp tục...\n")
    
    try:
        # Gọi API
        bot_answer = ask_true_false(question)
        
        # Kiểm tra lỗi
        if bot_answer.startswith("[ERROR]"):
            error_count += 1
        
        # Đánh giá
        score = evaluate_true_false_answer(bot_answer, ground_truth_letter)
        
    except Exception as e:
        bot_answer = f"[ERROR] {str(e)}"
        score = 0
        error_count += 1
        print(f"\n❌ Lỗi không xử lý được: {e}")
    
    # Lưu kết quả
    results.append({
        "question": question,
        "ground_truth_letter": ground_truth_letter,
        "ground_truth_text": ground_truth_text,
        "ground_truth_label": ground_truth_label,
        "bot_answer": bot_answer[:500] if bot_answer else "",
        "rel_type": rel_type,
        "score": score
    })
    
    # Lưu checkpoint sau mỗi batch
    if (idx + 1) % BATCH_SIZE == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8-sig')
        correct_so_far = sum(r['score'] for r in results)
        acc_so_far = correct_so_far / len(results) * 100
        print(f"💾 Checkpoint: {len(results)} câu | Accuracy: {acc_so_far:.1f}% | Errors: {error_count}")
    
    # Delay giữa các request trong batch
    if (idx + 1) % BATCH_SIZE != 0:
        time.sleep(DELAY_BETWEEN_REQUESTS)

# Lưu checkpoint cuối cùng
pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8-sig')

# Tổng kết
total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"🎉 HOÀN THÀNH ĐÁNH GIÁ - LEVEL 2!")
print(f"{'='*60}")
print(f"✅ Tổng số câu: {len(results)}")
print(f"⚠️ Số lỗi: {error_count}")
print(f"⏱️ Tổng thời gian: {total_time/60:.1f} phút ({total_time/3600:.2f} giờ)")
print(f"💾 Đã lưu vào: {CHECKPOINT_CSV}")

## 7. Tính toán và Hiển thị Metrics

In [ ]:
# Chuyển kết quả sang DataFrame
results_df = pd.DataFrame(results)

# === TÍNH TOÁN METRICS ===
total = len(results_df)
correct = results_df['score'].sum()

# y_true và y_pred cho metrics
y_true = results_df['ground_truth_label'].tolist()
y_pred = results_df['score'].tolist()

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Precision, Recall, F1 (binary)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# === HIỂN THỊ KẾT QUẢ ===
print("=" * 60)
print("📊 KẾT QUẢ ĐÁNH GIÁ GEMINI 2.5 FLASH - LEVEL 2")
print("=" * 60)
print(f"📁 Dataset: {INPUT_CSV}")
print(f"🤖 Model: {MODEL_NAME}")
print(f"📝 Tổng số câu hỏi: {total}")
print(f"✅ Số câu đúng: {correct}")
print(f"❌ Số câu sai: {total - correct}")
print("-" * 60)
print(f"🎯 ACCURACY:  {accuracy * 100:.2f}%")
print(f"📐 PRECISION: {precision * 100:.2f}%")
print(f"📏 RECALL:    {recall * 100:.2f}%")
print(f"⚖️  F1-SCORE:  {f1 * 100:.2f}%")
print("=" * 60)

# Tạo summary DataFrame
metrics_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Value': [f"{accuracy*100:.2f}%", f"{precision*100:.2f}%", f"{recall*100:.2f}%", f"{f1*100:.2f}%"]
})
metrics_summary

## 8. Phân tích theo Loại Quan hệ

In [ ]:
# Tính accuracy theo từng loại quan hệ
rel_stats = results_df.groupby('rel_type').agg({
    'score': ['sum', 'count', 'mean']
}).round(4)
rel_stats.columns = ['Correct', 'Total', 'Accuracy']
rel_stats['Accuracy'] = (rel_stats['Accuracy'] * 100).round(2)
rel_stats = rel_stats.sort_values('Accuracy', ascending=False)

print("📊 ACCURACY THEO LOẠI QUAN HỆ (GEMINI - LEVEL 2):")
print("-" * 50)
rel_stats

In [ ]:
# Vẽ biểu đồ Accuracy theo loại quan hệ
fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.RdYlGn(rel_stats['Accuracy'] / 100)
bars = ax.barh(rel_stats.index, rel_stats['Accuracy'], color=colors)

# Thêm label trên mỗi bar
for bar, acc in zip(bars, rel_stats['Accuracy']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
            f'{acc:.1f}%', va='center', fontsize=10)

ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_ylabel('Relationship Type', fontsize=12)
ax.set_title(f'📊 Accuracy theo Loại Quan hệ - {MODEL_NAME} - Level 2 (True/False)', fontsize=14, fontweight='bold')
ax.set_xlim(0, 110)
ax.axvline(x=accuracy*100, color='red', linestyle='--', label=f'Overall Avg: {accuracy*100:.1f}%')
ax.legend()

plt.tight_layout()
plt.savefig('gemini_level2_accuracy_by_reltype.png', dpi=150)
plt.show()

## 9. Phân tích theo Đáp án Ground Truth (A vs B)

In [ ]:
# Phân tích accuracy theo đáp án ground truth
answer_stats = results_df.groupby('ground_truth_letter').agg({
    'score': ['sum', 'count', 'mean']
}).round(4)
answer_stats.columns = ['Correct', 'Total', 'Accuracy']
answer_stats['Accuracy'] = (answer_stats['Accuracy'] * 100).round(2)

print("📊 ACCURACY THEO ĐÁP ÁN GROUND TRUTH (GEMINI):")
print("-" * 50)
print("A = Đúng (True), B = Sai (False)")
print("-" * 50)
answer_stats

In [ ]:
# Confusion Matrix
# Tính confusion matrix dựa trên bot choice vs ground truth
print("📊 CONFUSION MATRIX:")
print("-" * 50)
print("(Rows = Ground Truth, Cols = Predicted)")

# Tính số lượng cho từng trường hợp
true_a = results_df[results_df['ground_truth_letter'] == 'A']
true_b = results_df[results_df['ground_truth_letter'] == 'B']

# True A, Predicted Correct (A)
ta_pa = true_a['score'].sum()
# True A, Predicted Wrong (B)
ta_pb = len(true_a) - ta_pa
# True B, Predicted Correct (B)
tb_pb = true_b['score'].sum()
# True B, Predicted Wrong (A)
tb_pa = len(true_b) - tb_pb

print(f"\n             Predicted A    Predicted B")
print(f"True A (Đúng)    {ta_pa:>5}         {ta_pb:>5}")
print(f"True B (Sai)     {tb_pa:>5}         {tb_pb:>5}")

## 10. Lưu kết quả

In [ ]:
# Lưu kết quả chi tiết
results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"💾 Đã lưu kết quả chi tiết vào: {OUTPUT_CSV}")

# Lưu metrics summary
metrics_full = {
    'metric': ['accuracy', 'precision', 'recall', 'f1_score', 'total_questions', 'correct_answers', 'error_count'],
    'value': [accuracy, precision, recall, f1, total, correct, error_count]
}
pd.DataFrame(metrics_full).to_csv('eval_gemini_level2_metrics.csv', index=False)
print("💾 Đã lưu metrics vào: eval_gemini_level2_metrics.csv")

## 11. Phân tích một số câu trả lời

In [ ]:
# Xem một số câu đúng
print("✅ MẪU CÂU TRẢ LỜI ĐÚNG:")
print("=" * 80)
correct_samples = results_df[results_df['score'] == 1].head(3)

for idx, row in correct_samples.iterrows():
    print(f"\n📌 Câu hỏi: {row['question']}")
    print(f"   ✅ Đáp án đúng: {row['ground_truth_letter']} - {row['ground_truth_text'][:100]}...")
    print(f"   🤖 Gemini: {row['bot_answer'][:200]}...")
    print("-" * 80)

In [ ]:
# Phân tích các câu trả lời sai
wrong_answers = results_df[results_df['score'] == 0]
print(f"\n❌ PHÂN TÍCH CÂU TRẢ LỜI SAI ({len(wrong_answers)} câu):")
print("=" * 80)

# Hiển thị 5 câu sai đầu tiên
for idx, row in wrong_answers.head(5).iterrows():
    print(f"\n📌 Câu hỏi: {row['question']}")
    print(f"   ✅ Đáp án đúng: {row['ground_truth_letter']} - {row['ground_truth_text'][:100]}...")
    print(f"   ❌ Gemini: {row['bot_answer'][:200]}...")
    print(f"   🏷️ Loại: {row['rel_type']}")
    print("-" * 80)

## 12. Tổng kết và So sánh

In [ ]:
# Tổng kết cuối cùng
print("\n" + "=" * 60)
print("🏆 TỔNG KẾT ĐÁNH GIÁ GEMINI 2.5 FLASH - LEVEL 2")
print("=" * 60)
print(f"""
🤖 Model: {MODEL_NAME}
📁 Dataset: {INPUT_CSV}
📝 Số câu hỏi: {total}
🎯 Loại câu hỏi: Đúng/Sai (True/False)
⚠️ Số lỗi: {error_count}

┌──────────────┬─────────────┐
│ Metric       │ Value       │
├──────────────┼─────────────┤
│ Accuracy     │ {accuracy*100:>8.2f}%  │
│ Precision    │ {precision*100:>8.2f}%  │
│ Recall       │ {recall*100:>8.2f}%  │
│ F1-Score     │ {f1*100:>8.2f}%  │
└──────────────┴─────────────┘

📈 Loại quan hệ tốt nhất: {rel_stats.index[0]} ({rel_stats['Accuracy'].iloc[0]:.1f}%)
📉 Loại quan hệ kém nhất: {rel_stats.index[-1]} ({rel_stats['Accuracy'].iloc[-1]:.1f}%)

✅ Đánh giá Level 2 hoàn tất!

📋 SO SÁNH CÁC MÔ HÌNH:
┌────────────────────────┬─────────────────────────────────┐
│ Model                  │ File Metrics                    │
├────────────────────────┼─────────────────────────────────┤
│ Iterative RAG - L1     │ eval_level1_metrics.csv         │
│ Gemini 2.5 - L1        │ eval_gemini_level1_metrics.csv  │
│ Iterative RAG - L2     │ eval_level2_metrics.csv         │
│ Gemini 2.5 - L2        │ eval_gemini_level2_metrics.csv  │
└────────────────────────┴─────────────────────────────────┘
""")